# Issue #7 — capacity-weighted regional weather

This notebook reproduces the regional weather pipeline. It keeps one national GridToEV model while giving that model six regional views of the weather driving Irish wind and solar production.

## Leakage rule

An archived ECMWF run is not assumed available at its initialization time. GridToEV adds a conservative six-hour computation/publication delay and selects a run only when its publication time is no later than the model issue time. The delayed historical-forecast analysis proxy follows the same six-hour guard before it can contribute an error feature.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists():
    REPO_ROOT = REPO_ROOT.parent
PROCESSED = REPO_ROOT / 'data' / 'processed'
BENCHMARK = REPO_ROOT / 'benchmarks' / 'regional_weather_signals'
REPO_ROOT.name

'GridToEv-issue-7'

## Rebuild cached/downloaded sources and features

The script is resumable: downloaded JSON files are checksum-verified and reused. Raw responses stay in the Git-ignored data tree; normalized vintages and their provenance are committed.

In [2]:
environment = os.environ.copy()
environment['PYTHONPATH'] = str(REPO_ROOT / 'src')
completed = subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'build_regional_weather_data.py'),
        '--workers',
        '4',
    ],
    cwd=REPO_ROOT,
    env=environment,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

{
  "model_runs": 120,
  "regions": 6,
  "forecast_rows": 10080,
  "analysis_rows": 4608,
  "feature_rows": 2867,
  "feature_columns": 65,
  "ablation_status": "rejected_development_gate",
  "production_decision": "exclude",
  "mean_mae_improvement_fraction": 6.266931215729785e-05
}



## Reproducible regional proxy weights

Wind and solar are normalized separately because their geographic distributions differ. These are transparent engineering proxies, not an official asset register. Each region has a missing flag and the output records represented-capacity coverage.

In [3]:
weights = pd.read_csv(REPO_ROOT / 'config' / 'regional_weather_capacity_weights.csv')
weights[['region', 'latitude', 'longitude', 'wind_capacity_mw', 'solar_capacity_mw']]

,region,latitude,longitude,wind_capacity_mw,solar_capacity_mw
0,northwest,54.80,-7.80,1100,90
1,west,53.35,-9.05,900,120
2,southwest,52.20,-9.70,1400,250
3,midlands,53.20,-7.70,600,400
4,east,53.35,-6.30,300,450
5,southeast,52.30,-6.50,700,450


## Coverage and causal quality gates

In [4]:
quality = json.loads((PROCESSED / 'regional_weather_quality_report.json').read_text())
pd.Series({
    'forecast_runs': quality['model_run_count'],
    'forecast_vintage_rows': quality['forecast_vintage_rows'],
    'analysis_proxy_rows': quality['analysis_proxy_rows'],
    'feature_rows': quality['rows'],
    'forecast_coverage': quality['forecast_source_coverage'],
    'causal_error_coverage': quality['error_source_coverage'],
    'future_publication_violations': quality['future_publication_violations'],
    'duplicate_keys': quality['duplicate_natural_keys'],
})

forecast_runs                      120.000000
forecast_vintage_rows            10080.000000
analysis_proxy_rows               4608.000000
feature_rows                      2867.000000
forecast_coverage                    1.000000
causal_error_coverage                0.991629
future_publication_violations        0.000000
duplicate_keys                       0.000000
dtype: float64

In [5]:
features = pd.read_csv(PROCESSED / 'regional_weather_features_30_60.csv')
features[[
    'issue_timestamp_utc',
    'forecast_horizon_minutes',
    'weather_wind_weighted_speed_100m_mps',
    'weather_solar_weighted_cloud_cover_pct',
    'weather_wind_error_mae_24h_mps',
]].head()

,issue_timestamp_utc,forecast_horizon_minutes,weather_wind_weighted_speed_100m_mps,weather_solar_weighted_cloud_cover_pct,weather_wind_error_mae_24h_mps
0,2026-01-02T00:00:00Z,30,8.5496,68.039773,NaN
1,2026-01-02T00:00:00Z,60,8.5546,68.301136,NaN
2,2026-01-02T00:30:00Z,30,8.5546,68.301136,NaN
3,2026-01-02T00:30:00Z,60,8.5138,63.823864,NaN
4,2026-01-02T01:00:00Z,30,8.5138,63.823864,NaN


## Six-variant development ablation

Model selection uses three expanding training folds and validation only. The final 15% test period stays sealed. Weather enters production only after at least 15% mean per-fold MAE improvement with no degrading fold.

In [6]:
ablation = json.loads((BENCHMARK / 'ablation_report.json').read_text())
pd.DataFrame([
    {
        'variant': name,
        'features': result['feature_count'],
        'mean_improvement': result['mean_mae_improvement_fraction'],
        'worst_fold': result['worst_fold_improvement_fraction'],
    }
    for name, result in ablation['variants'].items()
]).sort_values('mean_improvement', ascending=False)

,variant,features,mean_improvement,worst_fold
3,causal_errors_only,8,0.000063,-0.021945
5,solar_aggregate_only,4,-0.008072,-0.025983
1,aggregate_forecast_and_errors,23,-0.037401,-0.113029
2,aggregate_forecast_only,15,-0.051897,-0.121141
4,wind_aggregate_only,12,-0.065323,-0.248700
0,all_weather,59,-0.073116,-0.226157


In [7]:
pd.Series({
    'selected_variant': ablation['selected_variant'],
    'production_decision': ablation['production_decision'],
    'mean_improvement': ablation['metrics']['mean_mae_improvement_fraction'],
    'worst_fold': ablation['metrics']['worst_fold_improvement_fraction'],
    'final_test_accessed': ablation['final_test_accessed'],
})

selected_variant       causal_errors_only
production_decision               exclude
mean_improvement                 0.000063
worst_fold                      -0.021945
final_test_accessed                 False
dtype: object